# Phase 0c: τ-bench pilot with Qwen3-32B on A100

**Why this notebook:** Qwen3-8B passed 0 of 13 valid runs over 7 airline tasks (see `02_TauBench_vLLM_Pilot.ipynb`). That is a floor effect, so it can't show if KiL helps. Here I try a much stronger model.

**Setup**
- GPU: **A100** (Runtime → Change runtime type)
- Agent: `Qwen/Qwen3-32B-AWQ` (4-bit) on vLLM, thinking **on**
- User simulator: the same model on the same server, thinking **off**
- Domain: airline, all 50 tasks, one try each (k=1), 4 tasks at a time
- System: stock agent only (the baseline)

**Changes from 02**
- Bigger model: Qwen3-32B instead of Qwen3-8B.
- The user simulator also moves to Qwen3-32B. Note it in the report.
- Each agent call can write at most 8192 tokens, and may take up to 30 min. In 02, thinking sometimes ran past 600s and the task died.
- An empty agent reply (thinking only) now counts as a normal fail. Before, it crashed τ-bench's scoring.
- `summarize.py` now labels how each task ended more carefully.

**Result in one line:** pass^1 = 0.22 on all 50 tasks (11/49 valid), 0.7 min per task. Qwen3-32B is the main study model. Details in the Summary at the end.

**How to run:** Run the cells one by one, top to bottom. Don't use Run all after the server is up: each run of the server cell starts a new server and fills the GPU.

In [ ]:
# check which GPU I got
# this notebook needs an A100 (40GB or 80GB)
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(out)
gpu_name, mem_mb = out.split(", ")
assert int(mem_mb) >= 40000, "Need an A100. Runtime → Change runtime type → A100."

MODEL = "Qwen/Qwen3-32B-AWQ"   # 4-bit, about 20GB of weights
SERVED = "qwen3-32b"           # the short name tau-bench uses
MAX_LEN = 32768
print("model:", MODEL, "| max length:", MAX_LEN)

NVIDIA A100-SXM4-40GB, 40960
model: Qwen/Qwen3-32B-AWQ | max length: 32768


In [ ]:
# mount Google Drive so my results are saved
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# go into the repo and install everything
# Colab drops packages on restart, so I run this every time
%cd /content/drive/MyDrive/cmsc691-10-project2
!pip install -q -e .
# new litellm needs mcp, tau-bench does not list it
!pip install -q mcp
# vLLM is big, this takes a few minutes
!pip install -q vllm
print("installed")

/content/drive/MyDrive/cmsc691-10-project2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for tau_bench (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.5 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf

In [ ]:
# count airline test tasks without importing tau-bench
src = open("/content/drive/MyDrive/cmsc691-10-project2/tau_bench/envs/airline/tasks_test.py").read()
print(f"{src.count('    Task(')} tasks")

50 tasks


In [ ]:
# start the vLLM server in the background
# run this cell ONCE; each run starts another server
# hermes parser: lets Qwen call tools
# qwen3 reasoning parser: keeps the thinking text out of the reply
import subprocess
cmd = ["vllm", "serve", MODEL,
       "--served-model-name", SERVED,
       "--max-model-len", str(MAX_LEN),
       "--gpu-memory-utilization", "0.90",
       "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
       "--reasoning-parser", "qwen3",
       "--port", "8000"]
server = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
print("server starting, pid", server.pid)

server starting, pid 2982


In [ ]:
# wait until the server is ready
# first time it also downloads the model, ~20GB
import time, requests
for i in range(120):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=5).ok:
            print(f"ready after {i*10}s")
            break
    except Exception:
        pass
    if server.poll() is not None:
        print("server died, real error:")
        break
    time.sleep(10)
else:
    print("not ready after 20 min")
!grep -iE "error|out of memory" /content/vllm.log | tail -n 5

ready after 320s


In [ ]:
# smoke test: can Qwen call a tool?
import json
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
tools = [{"type": "function", "function": {
    "name": "get_reservation",
    "description": "Look up a reservation",
    "parameters": {"type": "object",
                   "properties": {"reservation_id": {"type": "string"}},
                   "required": ["reservation_id"]}}}]

r = client.chat.completions.create(
    model=SERVED,
    messages=[{"role": "user", "content": "Look up reservation ABC123."}],
    tools=tools,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}})

msg = r.choices[0].message
if msg.tool_calls:
    call = msg.tool_calls[0].function
    print("✅ tool call works")
    print("   tool:", call.name)
    print("   args:", json.loads(call.arguments))
else:
    print("❌ no tool call, Qwen replied with text:")
    print("  ", msg.content)

✅ tool call works
   tool: get_reservation
   args: {'reservation_id': 'ABC123'}


In [ ]:
%%writefile /content/drive/MyDrive/cmsc691-10-project2/pilot_vllm.py
# run tau-bench against my local vLLM server
import sys
import litellm

# hide litellm's "Give Feedback" lines, they flood the output
litellm.suppress_debug_info = True

# thinking on for the agent only
# the agent's calls carry tools, the user simulator's calls don't
AGENT_THINKING = True

# cap each agent reply, so long thinking can't hang a task
AGENT_MAX_TOKENS = 8192
CALL_TIMEOUT = 1800  # seconds

_real = litellm.completion

def set_thinking(*args, **kwargs):
    is_agent = bool(kwargs.get("tools"))
    think = AGENT_THINKING if is_agent else False
    # copy first, so I never change tau-bench's own dict
    extra = dict(kwargs.get("extra_body") or {})
    extra["chat_template_kwargs"] = {"enable_thinking": think}
    kwargs["extra_body"] = extra
    if is_agent:
        kwargs.setdefault("max_tokens", AGENT_MAX_TOKENS)
    kwargs["timeout"] = CALL_TIMEOUT
    # return _real(*args, **kwargs)
    res = _real(*args, **kwargs)
    # an empty reply (thinking only) crashes tau-bench's scoring
    # turn it into "" so it counts as a normal fail
    msg = res.choices[0].message
    if msg.content is None and not msg.tool_calls:
        msg.content = ""
    return res

# swap it in BEFORE tau-bench imports completion
litellm.completion = set_thinking

sys.argv[0] = "run.py"
from run import main  # run.py is in the repo folder
main()

Overwriting /content/drive/MyDrive/cmsc691-10-project2/pilot_vllm.py


In [ ]:
%%writefile /content/drive/MyDrive/cmsc691-10-project2/summarize.py
# summarize one tau-bench result file
# API errors are not real failures, so I leave them out of pass^1
# I also show how each task ended, to spot loops
import json, sys, textwrap

rows = sorted(json.load(open(sys.argv[1])), key=lambda r: r["task_id"])
err = [r for r in rows if "error" in (r.get("info") or {})]
ok = [r for r in rows if r not in err]
passed = sum(r["reward"] == 1 for r in ok)

def ended(traj):
    # 30 steps = about 60+ messages: it ran out of steps (a loop)
    if len(traj) >= 60:
        return "STEP LIMIT"
    last = (traj[-1].get("content") or "") if traj else ""
    if "###STOP###" in last:
        return "user stop"
    # the agent ended it, e.g. by handing the user to a human
    return "agent end"

print(f"runs {len(rows)} | API errors {len(err)} | valid {len(ok)} | passed {passed}")
if ok:
    print(f"pass^1 = {passed / len(ok):.2f}")
print()
print(f"{'task':>4}  {'result':6}  {'msgs':>4}  {'tools':>5}  ended")
print("-" * 38)
for r in ok:
    traj = r["traj"]
    tools = sum(len(m.get("tool_calls") or []) for m in traj if m.get("role") == "assistant")
    res = "pass" if r["reward"] == 1 else "fail"
    print(f"{r['task_id']:>4}  {res:6}  {len(traj):>4}  {tools:>5}  {ended(traj)}")
for r in err:
    print(f"\nerror task {r['task_id']}:")
    print(textwrap.fill(r["info"]["error"], 120))

Overwriting /content/drive/MyDrive/cmsc691-10-project2/summarize.py


In [ ]:
# baseline pilot: airline, tasks 0-9, one try each
# agent and user: qwen3-32b on my vLLM server
# 4 tasks at a time, vLLM handles them together
# tau-bench reads the server address from these two env vars
import os, time
os.environ["HOSTED_VLLM_API_BASE"] = "http://localhost:8000/v1"
os.environ["HOSTED_VLLM_API_KEY"] = "none"

# TASKS = "0 1 2 3 4 5 6 7 8 9"
TASKS = " ".join(str(i) for i in range(50))
CONCURRENCY = 4

%cd /content/drive/MyDrive/cmsc691-10-project2
start = time.time()
!python pilot_vllm.py --env airline --agent-strategy tool-calling \
  --model {SERVED} --model-provider hosted_vllm \
  --user-model {SERVED} --user-model-provider hosted_vllm \
  --user-strategy llm --num-trials 1 \
  --temperature 0.0 --task-ids {TASKS} \
  --max-concurrency {CONCURRENCY} --log-dir results_vllm

# time per task, so I can estimate the full study
mins = (time.time() - start) / 60
n = len(TASKS.split())
print(f"\n⏱ {mins:.1f} min for {n} tasks = {mins / n:.1f} min per task (concurrency {CONCURRENCY})")

/content/drive/MyDrive/cmsc691-10-project2
Namespace(num_trials=1, env='airline', model='qwen3-32b', model_provider='hosted_vllm', user_model='qwen3-32b', user_model_provider='hosted_vllm', agent_strategy='tool-calling', temperature=0.0, task_split='test', start_index=0, end_index=-1, task_ids=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49], log_dir='results_vllm', max_concurrency=4, seed=10, shuffle=0, user_strategy='llm', few_shot_displays_path=None)
Loading user with strategy: llm
Running tasks [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49] (checkpoint path: results_vllm/tool-calling-qwen3-32b-0.0_range_0--1_user-qwen3-32b-llm_1026010150.json)
Running task 0
Running task 3
Running task 1
Running task 

In [ ]:
# read the newest result file
%cd /content/drive/MyDrive/cmsc691-10-project2
!python summarize.py $(ls -t results_vllm/*.json | head -1)

/content/drive/MyDrive/cmsc691-10-project2
runs 50 | API errors 1 | valid 49 | passed 11
pass^1 = 0.22

task  result  msgs  tools  ended
--------------------------------------
   0  fail      22      5  user stop
   1  fail      28      7  user stop
   2  fail      58     24  user stop
   3  fail      30     10  user stop
   4  fail      18      3  user stop
   5  fail      14      4  agent end
   6  fail      20      5  user stop
   8  fail      34     10  user stop
   9  fail      44     12  user stop
  10  fail      34     13  agent end
  11  fail      20      5  user stop
  12  pass      12      3  agent end
  13  fail      42     13  agent end
  14  fail      24      5  user stop
  15  pass       4      1  agent end
  16  pass      32     11  user stop
  17  fail      32      8  user stop
  18  pass       8      2  agent end
  19  fail      16      5  user stop
  20  pass      14      3  user stop
  21  fail      18      5  user stop
  22  fail      20      6  user stop
  23  fail

In [ ]:
# show the last 12 messages of one task, short
TASK_ID = 0
import json, glob, os
f = max(glob.glob("/content/drive/MyDrive/cmsc691-10-project2/results_vllm/*.json"), key=os.path.getmtime)
row = next(r for r in json.load(open(f)) if r["task_id"] == TASK_ID)
for m in row["traj"][-12:]:
    role = m["role"]
    text = (m.get("content") or "")[:150].replace("\n", " ")
    calls = [c["function"]["name"] for c in (m.get("tool_calls") or [])]
    print(f"{role:9} | {text} {calls if calls else ''}")

assistant |  ['get_user_details']
tool      | Error: user not found 
assistant |   I cannot proceed with the booking until I have your user ID. Could you please provide your user ID (e.g., "sara_doe_496") so I can verify your profi 
user      | My user ID is mia_li_3668. Please continue with the booking. 
assistant |  ['get_user_details']
tool      | {"name": {"first_name": "Mia", "last_name": "Li"}, "address": {"address1": "975 Sunset Drive", "address2": "Suite 217", "city": "Austin", "country": " 
assistant |   To proceed with booking your selected flights (HAT136 and HAT039), I need the following details:  ---  ### **1. Passenger Information**   Your profi 
user      | I am traveling with only Amelia Ahmed. I would like to add no extra baggage. I do not want to purchase travel insurance. For payment, please use the $ 
assistant |  ['book_reservation']
tool      | {"reservation_id": "HATHAT", "user_id": "mia_li_3668", "origin": "JFK", "destination": "SEA", "flight_type": "one_way", "

## Summary

**What I did**
- Moved to Qwen3-32B (4-bit AWQ) on an A100 40GB. Agent and user simulator both use it.
- Agent thinking on, user thinking off. Each agent reply is capped at 8192 tokens; each call may take up to 30 min.
- Fixed a crash: an empty agent reply (thinking only) broke τ-bench's scoring. Empty replies now count as normal fails.
- Ran the baseline twice: tasks 0–9, then all 50 airline tasks.

**Results**
- GPU: A100 40GB. Server ready after about 5 min.
- Run 1, tasks 0–9: 2/9 valid passed (pass^1 0.22). 1 crash on an empty reply, fixed after. 12.5 min, 1.2 min per task.
- Run 2, all 50 tasks: **11/49 valid passed, pass^1 = 0.22.** 1 API error. 34.5 min, **0.7 min per task** at concurrency 4.
- Passed: tasks 12, 15, 16, 18, 20, 27, 35, 36, 38, 41, 45.
- API error: task 7. The dialogue grew past the 32768-token limit.
- How tasks ended: 4 hit the step limit (30, 42, 46, 48); the rest ended normally (user stop or agent end).

**Analysis**
- The floor effect is gone. 0.22 is well above 0 and far from 1, so KiL has room to help.
- Results change between runs even at temperature 0. Task 0 passed in run 1 and failed in run 2. This is the inconsistency my study is about, and pass^k will show it.
- Common errors in the dialogues: the agent made up a user ID ("sara_doe_496") before asking; it got booking details wrong (cabin, payment). These are the rule and argument errors KiL targets.
- Tasks 42 and 48 hit the step limit with only 2 tool calls. Agent and user just kept talking. Worth a look later.
- Time: 2000 runs × 0.7 min ≈ 23 hours on A100. Doable over a few sessions.

**Decision**
- Main study model: **Qwen3-32B-AWQ** for both agent and user.
- Fixed settings for all 5 systems: agent thinking on, user thinking off, 8192-token cap, 1800s timeout, empty reply = fail, temperature 0.0, max length 32768.
- If any of these change, the baseline must be rerun.
- This 50-task run is the baseline at k=1.

**Next steps**
1. Start Milestone 1: build the airline policy graph by hand from `tau_bench/envs/airline/wiki.md`.
2. Run the baseline with 8 trials (k=1..8) to get its full pass^k curve.
3. Build the runtime validator from the policy graph.
4. Then the Shallow and Semi-deep KiL systems.

## Update (Oct 28): switched to Gemini
Following the project requirements, I decided to use Gemini from 05 on, like the paper
"τ-bench: A Benchmark for Tool-Agent-User Interaction in Real-World Domains" (Yao et al., 2024).
01–03 stay as the record of how I chose a model (Groq, Qwen3-8B, Qwen3-32B).
04 builds the policy graph and does not depend on the model.
The main runs are in the Gemini notebooks: 05G_Gemini_Baseline, 06G_Gemini_Validator, and so on.
Model: `gemini-3.1-flash-lite` on the paid Gemini API.
Why this model: Colab's built-in Gemini and the free API key had too little quota
(about 20 dialogues, 20 requests/day), and `gemini-2.5-flash-lite` is closed to new users.